# 🦥 Fine-Tuning Meiyon Personal AI with Unsloth AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohilc/meiyonpersonalai/blob/main/notebooks/meiyon_unsloth_finetuning.ipynb)

**Unsloth** makes fine-tuning large language models like **Llama 3.2 (1B/3B)**, **Llama 3.1 (8B)**, and **Mistral 7B**:
- ⚡ **2x to 5x faster** training
- 💾 **70% to 80% less memory** consumption using custom Triton & CUDA kernels
- 🎯 **0% loss in accuracy** compared to vanilla HuggingFace / PyTorch
- 📦 **1-Click GGUF export** for instant local inference on **Ollama** or llama.cpp

This notebook trains Meiyon on Mohil's executive persona, academic attendance rules, autonomous tool calling, and security verification RAG.

## 1. Install Unsloth and Dependencies
Select **Runtime -> Change runtime type -> T4 GPU (Free tier)** before running.

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

## 2. Load Base Model with Unsloth
We use **`unsloth/Llama-3.2-3B-Instruct`** in 4-bit (`load_in_4bit = True`) to fit comfortably within the 15GB VRAM of a free T4 GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detection (Float16 or Bfloat16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ Llama-3.2-3B loaded with Unsloth optimizations!")

## 3. Configure LoRA PEFT Adapters
Unsloth updates only 1-2% of model parameters via Low-Rank Adaptation (LoRA), preserving base reasoning while mastering Meiyon's persona.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # 60% less VRAM
    random_state = 3407,
)
print("✅ LoRA adapters attached successfully!")

## 4. Load & Format Meiyon Training Dataset
If running in Colab, you can either clone the repo or load the pre-packaged dataset below.

In [ ]:
# Clone repo to fetch latest training data if not present
import os
if not os.path.exists("data/training/meiyon_dataset_alpaca.json"):
    !git clone https://github.com/Mohilc/meiyonpersonalai.git
    dataset_path = "meiyonpersonalai/data/training/meiyon_dataset_alpaca.json"
else:
    dataset_path = "data/training/meiyon_dataset_alpaca.json"

from datasets import load_dataset
dataset = load_dataset("json", data_files=dataset_path, split="train")
print(f"Loaded {len(dataset)} training examples from Meiyon dataset!")

SYSTEM_PROMPT = (
    "You are Meiyon, an autonomous Personal AI Executive Assistant and trusted partner for Mohil. "
    "Mohil is an AI & Computer Science student at Mar Baselios Christian College of Engineering and Technology, "
    "specializing in autonomous drone robotics, real-time computer vision, and backend systems. "
    "Maintain a warm, encouraging, concise, and highly competent executive tone. "
    "When actions or queries require external systems, use the provided tools accurately with structured arguments."
)

def formatting_prompts_func(batch):
    instructions = batch["instruction"]
    inputs       = batch["input"]
    outputs      = batch["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        user_text = f"{instruction}\n{input_text}".strip() if input_text else instruction
        text = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            f"{SYSTEM_PROMPT}<|eot_id|>\n"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{user_text}<|eot_id|>\n"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{output}<|eot_id|>"
        )
        texts.append(text)
    return { "text" : texts }

formatted_dataset = dataset.map(formatting_prompts_func, batched = True)
print("Sample prompt template:\n", formatted_dataset[0]["text"][:300], "...")

## 5. Train with Unsloth SFTTrainer
Runs fast QLoRA training in ~10-15 minutes on a free Colab T4 GPU.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("🎉 Training completed successfully with Unsloth!")

## 6. Test the Fine-Tuned Model
Fast inference with `FastLanguageModel.for_inference(model)`.

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = (
    f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    f"{SYSTEM_PROMPT}<|eot_id|>\n"
    f"<|start_header_id|>user<|end_header_id|>\n\n"
    f"Check my college attendance and tell me if I need more classes.<|eot_id|>\n"
    f"<|start_header_id|>assistant<|end_header_id|>\n\n"
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
response = tokenizer.batch_decode(outputs)
print(response[0].split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip())

## 7. Export Model to GGUF for Ollama (Local Sub-50ms Inference)
Exporting directly to 4-bit GGUF (`q4_k_m`) allows running Meiyon locally on CPU or GPU with zero cloud latency.

In [ ]:
# Export to 4-bit GGUF for Ollama / llama.cpp
model.save_pretrained_gguf("meiyon_llama32_3b_gguf", tokenizer, quantization_method = "q4_k_m")
print("✅ Exported GGUF model to 'meiyon_llama32_3b_gguf'!")

# Optional: Save LoRA adapters
model.save_pretrained("meiyon_lora_adapter")
tokenizer.save_pretrained("meiyon_lora_adapter")
print("✅ Saved LoRA adapter!")

## 8. Download GGUF or Push to Hugging Face Hub

In [ ]:
# Push directly to Hugging Face Hub (enter token when prompted)
# model.push_to_hub_merged("mohilc/meiyon-llama-3.2-3b", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Download the GGUF file directly in Colab:
from google.colab import files
import glob
gguf_files = glob.glob("meiyon_llama32_3b_gguf/*.gguf")
if gguf_files:
    print(f"Found GGUF: {gguf_files[0]}. Downloading...")
    files.download(gguf_files[0])
else:
    print("GGUF directory ready.")